# 🎯 SAM-Prior Enhanced UNet - Few-Shot Learning
Load SAM, extract priors, train UNet in few-shot mode

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 55
N_SUPPORT = 3  # Few-shot: 3 support examples
N_QUERY = 1    # 1 query example

print(f"\nSettings:")
print(f"  Device: {DEVICE}")
print(f"  Classes: {NUM_CLASSES}")
print(f"  Few-shot: {N_SUPPORT} support + {N_QUERY} query")

In [ ]:
# ============ LOAD SAM ============
print("\n📥 Loading Segment Anything Model...\n")

try:
    from segment_anything import sam_model_registry
    print("Using local segment_anything")
except:
    print("Installing segment_anything...")
    import subprocess
    subprocess.check_call(['pip', 'install', '-q', 'git+https://github.com/facebookresearch/segment-anything.git'])
    from segment_anything import sam_model_registry

# Try to load checkpoint
sam_checkpoint = 'sam_vit_b.pth'
if not Path(sam_checkpoint).exists():
    print(f"⚠️  {sam_checkpoint} not found, downloading...")
    import urllib.request
    url = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    urllib.request.urlretrieve(url, sam_checkpoint)
    print(f"✅ Downloaded {sam_checkpoint}")
else:
    print(f"✅ Found {sam_checkpoint}")

# Load SAM
sam = sam_model_registry['vit_b'](checkpoint=sam_checkpoint)
sam = sam.to(DEVICE)
sam.eval()

print(f"✅ SAM loaded (ViT-B)")
print(f"   Encoder params: {sum(p.numel() for p in sam.image_encoder.parameters()):,}")

In [ ]:
# ============ LOAD DATA ============
print("\n📁 Loading data...\n")

# Load images
image_dir = Path('train-images')
available_images = sorted([int(p.stem) for p in image_dir.glob('*.png')])
print(f"Found {len(available_images)} images")

images = []
for idx in tqdm(available_images, desc='Loading images'):
    img_path = image_dir / f'{idx}.png'
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = (img / 255.0).astype(np.float32)
        images.append(img)

images_np = np.array(images)

# Load labels
labels_df = pd.read_csv('y_train.csv', index_col=0)
labels_df = labels_df.T
labels = labels_df.iloc[:len(images_np)].values
labels_2d = labels.reshape(-1, 256, 256)

print(f"Images: {images_np.shape}")
print(f"Labels: {labels_2d.shape}")
print(f"✅ Data loaded and aligned!")

In [ ]:
# ============ SAM ENCODER WRAPPER ============

class SAMEncoder:
    """Extract SAM embeddings from images"""
    def __init__(self, sam_model, device):
        self.sam = sam_model
        self.device = device
        self.image_encoder = sam_model.image_encoder
    
    @torch.no_grad()
    def embed_image(self, image_np):
        """Convert image to 3-channel and get SAM embedding"""
        # Convert grayscale to 3-channel
        if image_np.ndim == 2:
            image_3ch = np.stack([image_np] * 3, axis=0)
        else:
            image_3ch = image_np
        
        # Normalize to [0, 255] for SAM
        image_3ch = (image_3ch * 255).astype(np.uint8)
        
        # Convert to tensor and preprocess
        image_tensor = torch.from_numpy(image_3ch).float().to(self.device)
        image_tensor = image_tensor.unsqueeze(0)  # Add batch dim
        
        # Get embedding
        embedding = self.image_encoder(image_tensor)
        return embedding

sam_encoder = SAMEncoder(sam, DEVICE)
print("✅ SAM Encoder created")

In [ ]:
# ============ SAM-PRIOR ENHANCED UNET ============

class SAMPriorUNet(nn.Module):
    """UNet that uses SAM embeddings as priors"""
    def __init__(self, num_classes=55, base_channels=32):
        super().__init__()
        
        # Input: SAM embedding (1, 256, 64, 64) -> convert to 1x256x256
        self.embed_adapter = nn.Sequential(
            nn.Upsample(size=(256, 256), mode='bilinear', align_corners=False),
            nn.Conv2d(256, base_channels, 1),
            nn.ReLU(inplace=True)
        )
        
        # Encoder
        self.enc1 = self._conv_block(base_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = self._conv_block(base_channels, base_channels*2)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = self._conv_block(base_channels*2, base_channels*4)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self._conv_block(base_channels*4, base_channels*8)
        
        # Decoder
        self.upconv3 = nn.ConvTranspose2d(base_channels*8, base_channels*4, 2, stride=2)
        self.dec3 = self._conv_block(base_channels*8, base_channels*4)
        
        self.upconv2 = nn.ConvTranspose2d(base_channels*4, base_channels*2, 2, stride=2)
        self.dec2 = self._conv_block(base_channels*4, base_channels*2)
        
        self.upconv1 = nn.ConvTranspose2d(base_channels*2, base_channels, 2, stride=2)
        self.dec1 = self._conv_block(base_channels*2, base_channels)
        
        # Output
        self.final = nn.Conv2d(base_channels, num_classes, 1)
    
    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x, sam_embedding):
        # Use SAM embedding as prior
        prior = self.embed_adapter(sam_embedding)
        x = x + prior  # Fuse with input
        
        # Encoder with skip connections
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        
        b = self.bottleneck(self.pool3(e3))
        
        # Decoder
        d3 = self.dec3(torch.cat([self.upconv3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], 1))
        
        return self.final(d1)

model = SAMPriorUNet(num_classes=NUM_CLASSES)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ SAMPriorUNet created: {total_params:,} parameters")

In [ ]:
# ============ FEW-SHOT TRAINING LOOP ============

print("\n🚀 Few-Shot Training Start\n")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

num_images = len(images_np)
num_episodes = num_images // (N_SUPPORT + N_QUERY)

print(f"Total episodes: {num_episodes}")
print(f"Samples per episode: {N_SUPPORT} (support) + {N_QUERY} (query)\n")

history = {'support_loss': [], 'query_loss': [], 'query_dice': []}

# Few-shot training loop
for episode in range(min(10, num_episodes)):  # 10 episodes for speed
    # Sample support and query indices
    all_indices = np.random.permutation(num_images)[:N_SUPPORT + N_QUERY]
    support_indices = all_indices[:N_SUPPORT]
    query_indices = all_indices[N_SUPPORT:]
    
    # ===== SUPPORT PHASE (Adaptation) =====
    model.train()
    support_loss_epoch = 0
    
    for s_idx in support_indices:
        # Get image and SAM embedding
        img = images_np[s_idx]
        label = labels_2d[s_idx]
        
        # Get SAM embedding
        with torch.no_grad():
            sam_emb = sam_encoder.embed_image(img)
        
        # Prepare input
        img_tensor = torch.from_numpy(img[np.newaxis, np.newaxis, :, :]).float().to(DEVICE)
        label_tensor = torch.from_numpy(label).long().to(DEVICE).unsqueeze(0)
        
        # Forward
        logits = model(img_tensor, sam_emb)
        loss = loss_fn(logits, label_tensor)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        support_loss_epoch += loss.item()
    
    support_loss_epoch /= len(support_indices)
    
    # ===== QUERY PHASE (Evaluation) =====
    model.eval()
    query_loss_epoch = 0
    query_dices = []
    
    with torch.no_grad():
        for q_idx in query_indices:
            img = images_np[q_idx]
            label = labels_2d[q_idx]
            
            # Get SAM embedding
            sam_emb = sam_encoder.embed_image(img)
            
            # Prepare input
            img_tensor = torch.from_numpy(img[np.newaxis, np.newaxis, :, :]).float().to(DEVICE)
            label_tensor = torch.from_numpy(label).long().to(DEVICE).unsqueeze(0)
            
            # Forward
            logits = model(img_tensor, sam_emb)
            loss = loss_fn(logits, label_tensor)
            query_loss_epoch += loss.item()
            
            # Compute Dice
            pred = torch.argmax(logits, dim=1)[0].cpu().numpy()
            dice = (2 * (pred == label).sum()) / (pred.size + label.size)
            query_dices.append(dice)
    
    query_loss_epoch /= len(query_indices)
    query_dice_epoch = np.mean(query_dices)
    
    history['support_loss'].append(support_loss_epoch)
    history['query_loss'].append(query_loss_epoch)
    history['query_dice'].append(query_dice_epoch)
    
    if (episode + 1) % 2 == 0:
        print(f"Episode {episode+1:2d}/10 | "
              f"Support Loss: {support_loss_epoch:.4f} | "
              f"Query Loss: {query_loss_epoch:.4f} | "
              f"Query Dice: {query_dice_epoch:.4f}")

print(f"\n✅ Few-shot training complete!")

In [ ]:
# ============ VISUALIZATION ============

print("\n📊 Training History\n")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Support Loss
axes[0].plot(history['support_loss'], marker='o', linewidth=2)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Loss')
axes[0].set_title('Support Phase Loss')
axes[0].grid(True, alpha=0.3)

# Query Loss
axes[1].plot(history['query_loss'], marker='s', linewidth=2, color='orange')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Loss')
axes[1].set_title('Query Phase Loss')
axes[1].grid(True, alpha=0.3)

# Query Dice
axes[2].plot(history['query_dice'], marker='^', linewidth=2, color='green')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Dice Score')
axes[2].set_title('Query Phase Dice')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final Query Dice: {history['query_dice'][-1]:.4f}")

In [ ]:
# ============ PREDICTION VIEWER ============

def show_sam_prediction(model, img_idx, images_np, labels_2d, sam_encoder, device):
    """Display image with SAM prior, GT, and prediction"""
    
    model.eval()
    with torch.no_grad():
        # Get data
        img = images_np[img_idx]
        label = labels_2d[img_idx]
        
        # Get SAM embedding
        sam_emb = sam_encoder.embed_image(img)
        
        # Get prediction
        img_tensor = torch.from_numpy(img[np.newaxis, np.newaxis, :, :]).float().to(device)
        logits = model(img_tensor, sam_emb)
        pred = torch.argmax(logits, dim=1)[0].cpu().numpy()
        
        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # Original
        axes[0].imshow(img, cmap='gray')
        axes[0].set_title('Original Image', fontsize=13, fontweight='bold')
        axes[0].axis('off')
        
        # Ground truth
        im1 = axes[1].imshow(label, cmap='tab20', interpolation='nearest')
        axes[1].set_title('Ground Truth', fontsize=13, fontweight='bold')
        axes[1].axis('off')
        plt.colorbar(im1, ax=axes[1], fraction=0.046)
        
        # Prediction
        im2 = axes[2].imshow(pred, cmap='tab20', interpolation='nearest')
        axes[2].set_title('SAM+UNet Prediction', fontsize=13, fontweight='bold')
        axes[2].axis('off')
        plt.colorbar(im2, ax=axes[2], fraction=0.046)
        
        plt.suptitle(f'Image {img_idx} - SAM-Prior Enhanced Segmentation', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Show predictions
for idx in [0, 1]:
    print(f"\n{'='*60}\nImage {idx}\n{'='*60}")
    show_sam_prediction(model, idx, images_np, labels_2d, sam_encoder, DEVICE)

## 🎨 Change image index below to test different predictions

In [ ]:
# ============ INTERACTIVE VIEWER ============

image_idx = 0  # ← Change this (0-7)

print(f"\n🖼️  Showing SAM-enhanced prediction for image {image_idx}...\n")
show_sam_prediction(model, image_idx, images_np, labels_2d, sam_encoder, DEVICE)